# Enabling Manufacturing Intelligence with Amazon Nova Multimodal Embeddings

This notebook accompanies the AWS blog post of the same name. It builds two retrieval pipelines
for aerospace manufacturing documents and compares them head-to-head:

1. **Text-only baseline** — OCR each document with Amazon Nova Lite 2.0, embed the extracted text, and index it.
2. **Multimodal pipeline** — Embed images and document pages directly with Amazon Nova Multimodal Embeddings.

Both pipelines use **Amazon S3 Vectors** for vector storage and retrieval, and are evaluated on the same
26-query test set with standard retrieval metrics (Recall@K, Precision@K, MRR, NDCG@K) plus
generation quality via LLM-as-Judge.

**Prerequisites:**
- AWS account with Amazon Bedrock access in `us-east-1`
- Model access enabled for `amazon.nova-2-multimodal-embeddings-v1:0` and `us.amazon.nova-2-lite-v1:0`
- Python 3.10+, dependencies in `requirements.txt`
- The `dataset/` folder with manufacturing images and PDFs (included in this repo)

## 0. Setup

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import json
import os
import re
import glob
import base64
import boto3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from IPython.display import display, Image, HTML

from utils.embeddings import (
    MODEL_ID,
    generate_text_embedding,
    generate_image_embedding,
    generate_document_embedding,
    cosine_similarity,
)
from utils.pdf_utils import pdf_to_png
from utils.eval_metrics import (
    recall_at_k,
    mrr,
    ndcg_at_k,
    evaluate_query,
    evaluate_dataset,
)

REGION = "us-east-1"
os.environ["AWS_DEFAULT_REGION"] = REGION

bedrock_runtime = boto3.client("bedrock-runtime", region_name=REGION)
s3vectors = boto3.client("s3vectors", region_name=REGION)

# Get AWS account ID for unique bucket naming
sts = boto3.client("sts")
ACCOUNT_ID = sts.get_caller_identity()["Account"]

EMBEDDING_DIM = 1024
S3_VECTOR_BUCKET = f"manufacturing-mme-vectors-{ACCOUNT_ID}"
MME_INDEX = "manufacturing-multimodal"
TEXT_ONLY_INDEX = "manufacturing-text-only"

# Models
OCR_MODEL = "us.amazon.nova-2-lite-v1:0"
GENERATION_MODEL = "us.amazon.nova-2-lite-v1:0"
JUDGE_MODEL = "us.anthropic.claude-sonnet-4-5-20250929-v1:0"

print(f"Account:          {ACCOUNT_ID}")
print(f"Region:           {REGION}")
print(f"Embedding model:  {MODEL_ID}")
print(f"Embedding dim:    {EMBEDDING_DIM}")
print(f"Vector bucket:    {S3_VECTOR_BUCKET}")

## 1. Dataset Overview

The dataset contains synthetic aerospace manufacturing documents: 15 standalone technical images
(CAD diagrams, inspection reports, test plots, material specs) and 5 multi-page PDFs (assembly procedures,
test reports, engineering change notices, material certifications, non-conformance reports).

In [ ]:
image_files = sorted(glob.glob("dataset/*.png"))
pdf_files = sorted(glob.glob("dataset/*.pdf"))

print(f"Standalone images: {len(image_files)}")
for f in image_files:
    print(f"  {os.path.basename(f)}")

print(f"\nPDF documents: {len(pdf_files)}")
for f in pdf_files:
    print(f"  {os.path.basename(f)}")

In [ ]:
# Display sample images in a single row
from PIL import Image as PILImage

sample_paths = [
    "dataset/nozzle_assembly_diagram.png",
    "dataset/weld_inspection_report.png",
    "dataset/inconel718_fatigue_sn_curve.png",
]

fig, axes = plt.subplots(1, 3, figsize=(24, 8))
for ax, path in zip(axes, sample_paths):
    img = PILImage.open(path)
    ax.imshow(img)
    ax.set_title(os.path.basename(path).replace(".png", "").replace("_", " ").title(), fontsize=10)
    ax.axis("off")
plt.suptitle("Sample Manufacturing Documents", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("sample_documents_composite.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: sample_documents_composite.png")

In [ ]:
# Load evaluation dataset
with open("evals_dataset.json") as f:
    eval_data = json.load(f)

print(f"Evaluation queries: {len(eval_data)}")
print(f"Sample: {eval_data[0]['query']}")
print(f"  Relevant docs: {eval_data[0]['relevant_doc_ids']}")

## 2. Create S3 Vectors Bucket and Indexes

We create one vector bucket with two indexes: one for multimodal embeddings and one for text-only embeddings.

In [ ]:
# Create vector bucket
try:
    s3vectors.create_vector_bucket(vectorBucketName=S3_VECTOR_BUCKET)
    print(f"Created vector bucket: {S3_VECTOR_BUCKET}")
except Exception as e:
    print(f"Vector bucket: {e}")

# Create multimodal index
try:
    s3vectors.create_index(
        vectorBucketName=S3_VECTOR_BUCKET,
        indexName=MME_INDEX,
        dataType="float32",
        dimension=EMBEDDING_DIM,
        distanceMetric="cosine",
    )
    print(f"Created index: {MME_INDEX}")
except Exception as e:
    print(f"Multimodal index: {e}")

# Create text-only index
try:
    s3vectors.create_index(
        vectorBucketName=S3_VECTOR_BUCKET,
        indexName=TEXT_ONLY_INDEX,
        dataType="float32",
        dimension=EMBEDDING_DIM,
        distanceMetric="cosine",
    )
    print(f"Created index: {TEXT_ONLY_INDEX}")
except Exception as e:
    print(f"Text-only index: {e}")

## 3. Pipeline A: Multimodal Embeddings

Embed images directly and PDF pages as document images using Nova Multimodal Embeddings.
No text extraction needed — the model processes visual content natively.

### 3.1 Embed Standalone Images

In [ ]:
mme_embeddings = []

for img_path in tqdm(image_files, desc="Image embeddings"):
    img_name = os.path.splitext(os.path.basename(img_path))[0]
    embedding = generate_image_embedding(img_path, dim=EMBEDDING_DIM)
    mme_embeddings.append({
        "key": f"img-{img_name}",
        "embedding": embedding,
        "metadata": {
            "source_file": os.path.basename(img_path),
            "type": "image",
            "image_path": img_path,
        },
    })

print(f"Generated {len(mme_embeddings)} image embeddings.")

### 3.2 Embed PDF Pages as Document Images

In [ ]:
for pdf_path in pdf_files:
    pdf_name = os.path.splitext(os.path.basename(pdf_path))[0]
    page_dir = f"dataset/{pdf_name}_pages"
    page_images = pdf_to_png(pdf_path, output_dir=page_dir, dpi=200)

    for page_img in tqdm(page_images, desc=pdf_name):
        page_num = os.path.basename(page_img).replace("page_", "").replace(".png", "")
        embedding = generate_document_embedding(page_img, dim=EMBEDDING_DIM)
        mme_embeddings.append({
            "key": f"doc-{pdf_name}-page-{page_num}",
            "embedding": embedding,
            "metadata": {
                "source_pdf": pdf_name,
                "page_number": int(page_num),
                "type": "document_page",
                "image_path": page_img,
            },
        })

print(f"\nTotal multimodal embeddings: {len(mme_embeddings)}")

### 3.3 Ingest into S3 Vectors (Multimodal Index)

In [ ]:
BATCH_SIZE = 50
ingested = 0

for i in range(0, len(mme_embeddings), BATCH_SIZE):
    batch = mme_embeddings[i : i + BATCH_SIZE]
    vectors = [
        {
            "key": item["key"],
            "data": {"float32": item["embedding"]},
            "metadata": item["metadata"],
        }
        for item in batch
    ]
    s3vectors.put_vectors(
        vectorBucketName=S3_VECTOR_BUCKET,
        indexName=MME_INDEX,
        vectors=vectors,
    )
    ingested += len(batch)

print(f"Ingested {ingested} vectors into '{MME_INDEX}'.")

## 4. Pipeline B: Text-Only Baseline

Extract text from every image and PDF page using Amazon Nova Lite 2.0 as an OCR engine,
then embed the extracted text. This represents what a text-only retrieval system can achieve
on these visually rich documents.

### 4.1 OCR with Nova Lite 2.0

In [ ]:
def extract_text_with_nova(image_path, model_id=OCR_MODEL):
    """Extract all visible text from an image using Nova Lite 2.0."""
    with open(image_path, "rb") as f:
        img_bytes = f.read()

    fmt = "png" if image_path.lower().endswith(".png") else "jpeg"

    response = bedrock_runtime.converse(
        modelId=model_id,
        messages=[{
            "role": "user",
            "content": [
                {"image": {"format": fmt, "source": {"bytes": img_bytes}}},
                {"text": (
                    "Extract ALL visible text from this image exactly as it appears. "
                    "Include all numbers, labels, annotations, table contents, headers, "
                    "and footnotes. Preserve the structure (tables, lists, sections) as much as possible. "
                    "Return only the extracted text, no commentary."
                )},
            ],
        }],
        inferenceConfig={"maxTokens": 4096, "temperature": 0.0},
    )
    return response["output"]["message"]["content"][0]["text"]

### 4.2 Extract Text from All Documents

In [ ]:
extracted_texts = {}  # key -> {"text": str, "image_path": str, "type": str}

# Extract text from standalone images
for img_path in tqdm(image_files, desc="OCR images"):
    img_name = os.path.splitext(os.path.basename(img_path))[0]
    text = extract_text_with_nova(img_path)
    extracted_texts[f"img-{img_name}"] = {
        "text": text,
        "image_path": img_path,
        "type": "image",
    }

# Extract text from PDF pages
for pdf_path in pdf_files:
    pdf_name = os.path.splitext(os.path.basename(pdf_path))[0]
    page_dir = f"dataset/{pdf_name}_pages"
    # Pages were already converted in Section 3.2
    page_images = sorted(glob.glob(f"{page_dir}/page_*.png"))

    for page_img in tqdm(page_images, desc=f"OCR {pdf_name}"):
        page_num = os.path.basename(page_img).replace("page_", "").replace(".png", "")
        text = extract_text_with_nova(page_img)
        extracted_texts[f"doc-{pdf_name}-page-{page_num}"] = {
            "text": text,
            "image_path": page_img,
            "type": "document_page",
            "source_pdf": pdf_name,
            "page_number": int(page_num),
        }

print(f"\nExtracted text from {len(extracted_texts)} documents.")

# Save OCR results for reproducibility
with open("extracted_texts.json", "w") as f:
    json.dump(extracted_texts, f, indent=2)
print("Saved: extracted_texts.json")

In [ ]:
# Spot-check a few extracted texts
for key in list(extracted_texts.keys())[:3]:
    print(f"\n{'='*60}")
    print(f"Key: {key}")
    print(f"Text (first 300 chars):\n{extracted_texts[key]['text'][:300]}")

### 4.3 Generate Text-Only Embeddings and Ingest

In [ ]:
text_embeddings = []

for key, entry in tqdm(extracted_texts.items(), desc="Text embeddings"):
    text = entry["text"]
    if not text.strip():
        continue
    embedding = generate_text_embedding(text, dim=EMBEDDING_DIM)
    metadata = {k: v for k, v in entry.items() if k != "text"}
    metadata["text_preview"] = text[:200]
    text_embeddings.append({
        "key": key,
        "embedding": embedding,
        "metadata": metadata,
    })

print(f"Generated {len(text_embeddings)} text-only embeddings.")

# Ingest into text-only index
ingested = 0
for i in range(0, len(text_embeddings), BATCH_SIZE):
    batch = text_embeddings[i : i + BATCH_SIZE]
    vectors = [
        {
            "key": item["key"],
            "data": {"float32": item["embedding"]},
            "metadata": item["metadata"],
        }
        for item in batch
    ]
    s3vectors.put_vectors(
        vectorBucketName=S3_VECTOR_BUCKET,
        indexName=TEXT_ONLY_INDEX,
        vectors=vectors,
    )
    ingested += len(batch)

print(f"Ingested {ingested} vectors into '{TEXT_ONLY_INDEX}'.")

## 5. Retrieval Evaluation: Multimodal Embeddings

Run the 26 evaluation queries against the multimodal index and measure retrieval quality.

In [ ]:
def search_index(query_embedding, index_name, top_k=10):
    """Query an S3 Vectors index and return list of (key, distance) tuples."""
    response = s3vectors.query_vectors(
        vectorBucketName=S3_VECTOR_BUCKET,
        indexName=index_name,
        queryVector={"float32": query_embedding},
        topK=top_k,
        returnDistance=True,
        returnMetadata=True,
    )
    return response["vectors"]

In [ ]:
MAX_K = 10
K_VALUES = (3, 5, 10)

def run_retrieval_eval(index_name, label):
    """Run retrieval evaluation for all queries against a given index."""
    per_query = []
    for item in eval_data:
        query_embed = generate_text_embedding(item['query'], dim=EMBEDDING_DIM, purpose="GENERIC_RETRIEVAL")
        results = search_index(query_embed, index_name, top_k=MAX_K)
        retrieved_ids = [v["key"] for v in results]
        metrics = evaluate_query(retrieved_ids, item["relevant_doc_ids"], k_values=K_VALUES)
        metrics["query"] = item["query"]
        per_query.append(metrics)
    avg = evaluate_dataset([{k: v for k, v in r.items() if k != "query"} for r in per_query])
    # Filter to only recall, mrr, ndcg
    avg = {k: v for k, v in avg.items() if "precision" not in k}
    print(f"\n{label} — Average Retrieval Metrics:")
    for metric, value in sorted(avg.items()):
        print(f"  {metric:>15s}: {value:.4f}")
    return per_query, avg

print("Evaluating multimodal pipeline...")
mme_per_query, mme_avg = run_retrieval_eval(MME_INDEX, "Multimodal (MME)")

### 5.1 Retrieval Metrics Table

In [ ]:
# Build metrics table (MME only, no precision)
metrics_to_show = ["mrr"] + [f"{m}@{k}" for k in K_VALUES for m in ["recall", "ndcg"]]

rows = []
for metric in metrics_to_show:
    rows.append({"Metric": metric, "Score": mme_avg.get(metric, 0.0)})

metrics_df = pd.DataFrame(rows).round(4)
display(metrics_df.style.format({"Score": "{:.4f}"}).set_caption("Multimodal Retrieval Metrics"))

### 5.2 Retrieval Metrics Chart

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

x = np.arange(len(metrics_to_show))
vals = [mme_avg.get(m, 0) for m in metrics_to_show]

bars = ax.bar(x, vals, color="#023047", edgecolor="white", width=0.6)

ax.set_ylabel("Score")
ax.set_title("Multimodal Retrieval Metrics (Nova MME)")
ax.set_xticks(x)
ax.set_xticklabels(metrics_to_show, rotation=30, ha="right")
ax.set_ylim(0, 1.05)
ax.grid(axis="y", alpha=0.3)

for bar in bars:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
            f"{bar.get_height():.2f}", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.savefig("mme_retrieval_metrics.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: mme_retrieval_metrics.png")

## 6. Generation Evaluation: LLM-as-Judge

For each query, retrieve context from both indexes, generate an answer, and score it
against the ground truth using an LLM judge.

In [ ]:
def vector_key_to_image_path(key):
    """Map a vector key back to its source image file path."""
    if key.startswith("doc-"):
        parts = key[4:]
        page_match = re.search(r'-page-(\d+)$', parts)
        if page_match:
            page_num = page_match.group(1)
            doc_name = parts[:page_match.start()]
            return f"dataset/{doc_name}_pages/page_{page_num}.png"
    elif key.startswith("img-"):
        name = key[4:]
        return f"dataset/{name}.png"
    return None


def generate_answer_multimodal(query, retrieved_keys, model_id=GENERATION_MODEL):
    """Generate an answer by passing retrieved images as multimodal context."""
    content_blocks = []
    for key in retrieved_keys[:5]:
        img_path = vector_key_to_image_path(key)
        if img_path and os.path.exists(img_path):
            with open(img_path, "rb") as f:
                img_bytes = f.read()
            fmt = "png" if img_path.endswith(".png") else "jpeg"
            content_blocks.append({"text": f"Retrieved document {len(content_blocks)//2 + 1}:"})
            content_blocks.append({
                "image": {"format": fmt, "source": {"bytes": img_bytes}}
            })
    content_blocks.append({
        "text": (
            f"Review ALL images above carefully. The answer may appear in any of them. "
            f"Answer the following question concisely and precisely.\n\n"
            f"Question: {query}\n\nAnswer:"
        )
    })
    response = bedrock_runtime.converse(
        modelId=model_id,
        messages=[{"role": "user", "content": content_blocks}],
        inferenceConfig={"maxTokens": 500, "temperature": 0.1},
    )
    return response["output"]["message"]["content"][0]["text"]


def generate_answer_text_only(query, retrieved_vectors):
    """Generate an answer using only the OCR-extracted text as context."""
    context_parts = []
    for v in retrieved_vectors[:5]:
        meta = v.get("metadata", {})
        preview = meta.get("text_preview", "")
        if preview:
            context_parts.append(f"[{v['key']}]: {preview}")
    context = "\n\n".join(context_parts)
    prompt = (
        f"Use the following retrieved text excerpts to answer the question. "
        f"Answer concisely and precisely.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {query}\n\nAnswer:"
    )
    response = bedrock_runtime.converse(
        modelId=GENERATION_MODEL,
        messages=[{"role": "user", "content": [{"text": prompt}]}],
        inferenceConfig={"maxTokens": 500, "temperature": 0.1},
    )
    return response["output"]["message"]["content"][0]["text"]


def judge_correctness(query, generated_answer, ground_truth, model_id=JUDGE_MODEL):
    """Score generated answer vs ground truth on a 1-5 scale."""
    prompt = (
        "You are an evaluation judge. Score the generated answer compared to the ground truth "
        "on a scale of 1-5.\n\n"
        "1 = Completely wrong or irrelevant\n"
        "2 = Partially relevant but mostly incorrect\n"
        "3 = Somewhat correct but missing key information\n"
        "4 = Mostly correct with minor omissions\n"
        "5 = Fully correct and complete\n\n"
        f"Question: {query}\n"
        f"Ground Truth: {ground_truth}\n"
        f"Generated Answer: {generated_answer}\n\n"
        'Respond with ONLY a JSON object: {"score": <1-5>, "reason": "<brief explanation>"}'
    )
    response = bedrock_runtime.converse(
        modelId=model_id,
        messages=[{"role": "user", "content": [{"text": prompt}]}],
        inferenceConfig={"maxTokens": 200, "temperature": 0.0},
    )
    return response["output"]["message"]["content"][0]["text"]

In [ ]:
gen_eval_data = [item for item in eval_data if item.get("ground_truth_answer")]
print(f"Queries with ground truth: {len(gen_eval_data)}")
print(f"Generation model: {GENERATION_MODEL}")
print(f"Judge model:      {JUDGE_MODEL}\n")

mme_judge_scores = []
text_judge_scores = []

for i, item in enumerate(gen_eval_data):
    query = item["query"]
    gt = item["ground_truth_answer"]

    # Query embedding
    query_embed = generate_text_embedding(query, dim=EMBEDDING_DIM, purpose="GENERIC_RETRIEVAL")

    # --- Multimodal pipeline ---
    mme_results = search_index(query_embed, MME_INDEX, top_k=5)
    mme_keys = [v["key"] for v in mme_results]
    mme_answer = generate_answer_multimodal(query, mme_keys)
    mme_judgment = judge_correctness(query, mme_answer, gt)
    try:
        mme_score = int(re.search(r'"score"\s*:\s*(\d)', mme_judgment).group(1))
    except Exception:
        mme_score = None
    mme_judge_scores.append(mme_score)

    # --- Text-only pipeline ---
    text_results = search_index(query_embed, TEXT_ONLY_INDEX, top_k=5)
    text_answer = generate_answer_text_only(query, text_results)
    text_judgment = judge_correctness(query, text_answer, gt)
    try:
        text_score = int(re.search(r'"score"\s*:\s*(\d)', text_judgment).group(1))
    except Exception:
        text_score = None
    text_judge_scores.append(text_score)

    if (i + 1) % 5 == 0 or (i + 1) == len(gen_eval_data):
        print(f"Evaluated {i + 1}/{len(gen_eval_data)} queries")

print("\nGeneration evaluation complete.")

In [ ]:
# Summarize generation scores
valid_mme = [s for s in mme_judge_scores if s is not None]
valid_text = [s for s in text_judge_scores if s is not None]

print("Generation Quality (LLM-as-Judge)")
print("=" * 50)
print(f"  {'':30s} {'Text-Only':>10s} {'Multimodal':>10s}")
print(f"  {'Queries evaluated':30s} {len(valid_text):>10d} {len(valid_mme):>10d}")
print(f"  {'Average score (1-5)':30s} {np.mean(valid_text):>10.2f} {np.mean(valid_mme):>10.2f}")
print(f"  {'Normalized (0-1)':30s} {np.mean(valid_text)/5:>10.4f} {np.mean(valid_mme)/5:>10.4f}")

### 6.1 Generation Score Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

for ax, scores, title, color in [
    (axes[0], valid_text, "Text-Only (OCR)", "#8ECAE6"),
    (axes[1], valid_mme, "Multimodal (MME)", "#023047"),
]:
    bins = [0.5, 1.5, 2.5, 3.5, 4.5, 5.5]
    ax.hist(scores, bins=bins, color=color, edgecolor="white", rwidth=0.8)
    ax.set_title(f"{title}\nMean: {np.mean(scores):.2f}/5")
    ax.set_xlabel("Judge Score")
    ax.set_ylabel("Count")
    ax.set_xticks([1, 2, 3, 4, 5])
    ax.grid(axis="y", alpha=0.3)

plt.suptitle("Generation Quality: LLM-as-Judge Score Distribution", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("generation_score_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: generation_score_distribution.png")

## 7. Qualitative Examples

Examine specific queries where the multimodal pipeline outperforms text-only retrieval.

In [ ]:
# Find queries where MME score > text score
wins = []
for i, item in enumerate(gen_eval_data):
    ms = mme_judge_scores[i]
    ts = text_judge_scores[i]
    if ms is not None and ts is not None and ms > ts:
        wins.append((i, item["query"], ts, ms))

wins.sort(key=lambda x: x[3] - x[2], reverse=True)

print(f"Queries where multimodal outperformed text-only: {len(wins)}/{len(gen_eval_data)}\n")
for idx, query, ts, ms in wins[:5]:
    print(f"  Q: {query}")
    print(f"    Text-Only score: {ts}/5 | Multimodal score: {ms}/5  (delta: +{ms-ts})")
    print(f"    Relevant docs: {gen_eval_data[idx]['relevant_doc_ids']}")
    print()

In [ ]:
# Show detailed retrieval for top 3 wins
for idx, query, ts, ms in wins[:3]:
    item = gen_eval_data[idx]
    query_embed = generate_text_embedding(query, dim=EMBEDDING_DIM, purpose="GENERIC_RETRIEVAL")

    print(f"\n{'='*70}")
    print(f"Query: {query}")
    print(f"Expected: {item['relevant_doc_ids']}")
    print(f"Ground truth: {item['ground_truth_answer'][:150]}")

    # Text-only results
    text_results = search_index(query_embed, TEXT_ONLY_INDEX, top_k=3)
    print(f"\n  Text-Only Top-3 (score {ts}/5):")
    for r in text_results:
        hit = "HIT" if r["key"] in item["relevant_doc_ids"] else "miss"
        print(f"    [{hit}] {r['key']} (dist: {r.get('distance', 0):.4f})")

    # Multimodal results
    mme_results = search_index(query_embed, MME_INDEX, top_k=3)
    print(f"\n  Multimodal Top-3 (score {ms}/5):")
    for r in mme_results:
        hit = "HIT" if r["key"] in item["relevant_doc_ids"] else "miss"
        print(f"    [{hit}] {r['key']} (dist: {r.get('distance', 0):.4f})")

## 8. Save All Results

In [ ]:
results = {
    "config": {
        "embedding_dim": EMBEDDING_DIM,
        "embedding_model": MODEL_ID,
        "ocr_model": OCR_MODEL,
        "generation_model": GENERATION_MODEL,
        "judge_model": JUDGE_MODEL,
        "num_eval_queries": len(eval_data),
    },
    "retrieval": {
        "multimodal": {"per_query": mme_per_query, "averages": mme_avg},
    },
    "generation": {
        "multimodal": {
            "scores": mme_judge_scores,
            "avg_score": float(np.mean(valid_mme)) if valid_mme else None,
        },
        "text_only": {
            "scores": text_judge_scores,
            "avg_score": float(np.mean(valid_text)) if valid_text else None,
        },
    },
}

with open("evaluation_results.json", "w") as f:
    json.dump(results, f, indent=2, default=str)

print("Saved: evaluation_results.json")

## 9. Cleanup

Delete the S3 Vectors indexes and bucket to avoid ongoing storage costs.

**Uncomment the cells below when you are ready to clean up.**

In [ ]:
# # Delete indexes
# s3vectors.delete_index(vectorBucketName=S3_VECTOR_BUCKET, indexName=MME_INDEX)
# print(f"Deleted index: {MME_INDEX}")
#
# s3vectors.delete_index(vectorBucketName=S3_VECTOR_BUCKET, indexName=TEXT_ONLY_INDEX)
# print(f"Deleted index: {TEXT_ONLY_INDEX}")
#
# # Delete vector bucket
# s3vectors.delete_vector_bucket(vectorBucketName=S3_VECTOR_BUCKET)
# print(f"Deleted vector bucket: {S3_VECTOR_BUCKET}")
#
# print("\nCleanup complete.")